In [1]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
# data = np.load("radar_features_filtered_manual.npz")
data = np.load("radar_features_filtered.npz")

X = data["X"]
y = data["y"]
u = data["u"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [3]:
# data = np.load("radar_features_filtered_manual.npz")
data = np.load("radar_features_filtered.npz")

X = data["X"]
y = data["y"]
u = data["u"]

scaler = StandardScaler()
X = scaler.fit_transform(X)

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y_encoded,
    test_size=0.3,
    random_state=42,
    stratify=y_encoded
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

In [5]:
catboost = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    loss_function='MultiClass',
    verbose=100
)

catboost.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

0:	learn: 1.7346961	test: 1.7318978	best: 1.7318978 (0)	total: 86.6ms	remaining: 1m 26s
100:	learn: 0.4708784	test: 0.7991799	best: 0.7991799 (100)	total: 1.16s	remaining: 10.3s
200:	learn: 0.2167766	test: 0.5970061	best: 0.5970061 (200)	total: 2.25s	remaining: 8.95s
300:	learn: 0.1347801	test: 0.5263793	best: 0.5263793 (300)	total: 3.38s	remaining: 7.85s
400:	learn: 0.0951509	test: 0.4856700	best: 0.4856700 (400)	total: 4.39s	remaining: 6.55s
500:	learn: 0.0728869	test: 0.4612121	best: 0.4612121 (500)	total: 5.53s	remaining: 5.51s
600:	learn: 0.0583140	test: 0.4433465	best: 0.4432912 (599)	total: 6.55s	remaining: 4.35s
700:	learn: 0.0476383	test: 0.4294327	best: 0.4294327 (700)	total: 7.61s	remaining: 3.24s
800:	learn: 0.0395881	test: 0.4174141	best: 0.4174141 (800)	total: 8.66s	remaining: 2.15s
900:	learn: 0.0337324	test: 0.4086001	best: 0.4086001 (900)	total: 9.7s	remaining: 1.06s
999:	learn: 0.0292533	test: 0.4022485	best: 0.4020039 (993)	total: 10.7s	remaining: 0us

bestTest = 0.4

CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.05, loss_function='MultiClass', verbose=100)

In [6]:
val_pred = catboost.predict(X_val)
print("VALIDATION")
print(classification_report(y_val, val_pred))

VALIDATION
              precision    recall  f1-score   support

           0       1.00      0.89      0.94        19
           1       0.86      0.86      0.86        14
           2       0.95      1.00      0.98        41
           3       0.80      0.80      0.80        10
           4       0.00      0.00      0.00         3
           5       0.64      0.82      0.72        11

    accuracy                           0.89        98
   macro avg       0.71      0.73      0.72        98
weighted avg       0.87      0.89      0.88        98



In [7]:
test_pred = catboost.predict(X_test)
print("TEST")
print(classification_report(y_test, test_pred))

TEST
              precision    recall  f1-score   support

           0       0.90      0.90      0.90        20
           1       0.88      1.00      0.93        14
           2       0.91      0.98      0.94        41
           3       1.00      0.70      0.82        10
           4       1.00      0.50      0.67         2
           5       0.91      0.83      0.87        12

    accuracy                           0.91        99
   macro avg       0.93      0.82      0.86        99
weighted avg       0.91      0.91      0.91        99



In [8]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    catboost,
    X,
    y_encoded,
    cv=cv,
    scoring="f1_macro"
)

print("CV scores:", scores)
print("Mean F1:", scores.mean())

0:	learn: 1.7382335	total: 11.4ms	remaining: 11.4s
100:	learn: 0.4712117	total: 1.07s	remaining: 9.53s
200:	learn: 0.2292508	total: 2.13s	remaining: 8.49s
300:	learn: 0.1425684	total: 3.34s	remaining: 7.76s
400:	learn: 0.1036834	total: 4.4s	remaining: 6.58s
500:	learn: 0.0803488	total: 5.5s	remaining: 5.48s
600:	learn: 0.0641208	total: 6.55s	remaining: 4.35s
700:	learn: 0.0526893	total: 7.6s	remaining: 3.24s
800:	learn: 0.0438769	total: 8.68s	remaining: 2.16s
900:	learn: 0.0377186	total: 9.76s	remaining: 1.07s
999:	learn: 0.0327161	total: 10.8s	remaining: 0us
0:	learn: 1.7411924	total: 18.3ms	remaining: 18.3s
100:	learn: 0.4583002	total: 1.08s	remaining: 9.67s
200:	learn: 0.2137415	total: 2.29s	remaining: 9.09s
300:	learn: 0.1308919	total: 3.34s	remaining: 7.75s
400:	learn: 0.0912277	total: 4.42s	remaining: 6.61s
500:	learn: 0.0683455	total: 5.5s	remaining: 5.48s
600:	learn: 0.0541226	total: 6.57s	remaining: 4.36s
700:	learn: 0.0437004	total: 7.64s	remaining: 3.26s
800:	learn: 0.037094

In [9]:
# Use leave-one-user-out next
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(test_size=0.2, n_splits=1, random_state=42)

train_idx, test_idx = next(gss.split(X, y_encoded, groups=u))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

In [10]:
catboost.fit(X_train, y_train)

0:	learn: 1.7340186	total: 13.2ms	remaining: 13.2s
100:	learn: 0.4264307	total: 1.25s	remaining: 11.2s
200:	learn: 0.2049771	total: 2.45s	remaining: 9.74s
300:	learn: 0.1273110	total: 3.67s	remaining: 8.53s
400:	learn: 0.0880942	total: 4.86s	remaining: 7.26s
500:	learn: 0.0664909	total: 6.07s	remaining: 6.04s
600:	learn: 0.0521181	total: 7.38s	remaining: 4.9s
700:	learn: 0.0418450	total: 8.59s	remaining: 3.66s
800:	learn: 0.0346873	total: 9.78s	remaining: 2.43s
900:	learn: 0.0293811	total: 11s	remaining: 1.2s
999:	learn: 0.0254969	total: 12.1s	remaining: 0us


CatBoostClassifier(depth=6, iterations=1000, learning_rate=0.05, loss_function='MultiClass', verbose=100)

In [11]:
test_pred = catboost.predict(X_test)
print("TEST")
print(classification_report(y_test, test_pred))

TEST
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       0.77      0.96      0.86        25
           2       0.58      0.92      0.71        12
           3       0.60      0.67      0.63         9
           4       0.00      0.00      0.00         8
           5       0.77      0.48      0.59        21

    accuracy                           0.68        75
   macro avg       0.45      0.50      0.46        75
weighted avg       0.64      0.68      0.64        75



In [12]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

logo = LeaveOneGroupOut()

acc_scores = []
f1_scores = []

for train_idx, test_idx in logo.split(X, y_encoded, groups=u):

    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y_encoded[train_idx], y_encoded[test_idx]

    catboost.fit(X_train, y_train)

    y_pred = catboost.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')

    acc_scores.append(acc)
    f1_scores.append(f1)

    print(f"Fold Accuracy: {acc:.4f} | Fold F1: {f1:.4f}")

print("\n========== FINAL ==========")
print(f"Mean Accuracy: {np.mean(acc_scores):.4f}")
print(f"Mean Macro F1: {np.mean(f1_scores):.4f}")
print(f"Std Accuracy: {np.std(acc_scores):.4f}")
print(f"Std Macro F1: {np.std(f1_scores):.4f}")

0:	learn: 1.7425387	total: 11.6ms	remaining: 11.6s
100:	learn: 0.4423689	total: 1.28s	remaining: 11.4s
200:	learn: 0.2153299	total: 2.47s	remaining: 9.83s
300:	learn: 0.1338724	total: 3.71s	remaining: 8.6s
400:	learn: 0.0938797	total: 5.07s	remaining: 7.57s
500:	learn: 0.0711533	total: 6.29s	remaining: 6.26s
600:	learn: 0.0552166	total: 7.53s	remaining: 5s
700:	learn: 0.0452205	total: 8.79s	remaining: 3.75s
800:	learn: 0.0382519	total: 10s	remaining: 2.49s
900:	learn: 0.0324147	total: 11.3s	remaining: 1.25s
999:	learn: 0.0282197	total: 12.7s	remaining: 0us
Fold Accuracy: 0.5556 | Fold F1: 0.4041
0:	learn: 1.7328680	total: 12ms	remaining: 12s
100:	learn: 0.4385133	total: 1.31s	remaining: 11.6s
200:	learn: 0.2065913	total: 2.69s	remaining: 10.7s
300:	learn: 0.1283314	total: 3.97s	remaining: 9.21s
400:	learn: 0.0903342	total: 5.26s	remaining: 7.86s
500:	learn: 0.0688015	total: 6.54s	remaining: 6.51s
600:	learn: 0.0545883	total: 7.83s	remaining: 5.2s
700:	learn: 0.0445701	total: 9.17s	rema